# Weather Query Agent

This notebook demonstrates a simulated weather agent that fetches temperature for a given city.

## Overview

The weather query agent simulates an AI agent that:
- Maintains a database of weather information for various cities
- Processes natural language queries about weather
- Returns temperature and weather conditions for requested cities
- Handles unknown cities gracefully

## 1. Setup and Dependencies

In [ ]:
import random
from datetime import datetime
from typing import Dict, Optional, List
from dataclasses import dataclass
import re

## 2. Data Models

Define the data structures for weather information.

In [ ]:
@dataclass
class WeatherData:
    """Represents weather information for a city."""
    city: str
    country: str
    temperature_celsius: float
    temperature_fahrenheit: float
    condition: str
    humidity: int
    wind_speed_kmh: float
    timestamp: datetime
    
    def __str__(self) -> str:
        return (
            f"Weather in {self.city}, {self.country}:\n"
            f"  Temperature: {self.temperature_celsius}°C ({self.temperature_fahrenheit}°F)\n"
            f"  Condition: {self.condition}\n"
            f"  Humidity: {self.humidity}%\n"
            f"  Wind Speed: {self.wind_speed_kmh} km/h\n"
            f"  Last Updated: {self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}"
        )

## 3. Simulated Weather Database

Create a mock database with weather data for various cities around the world.

In [ ]:
class WeatherDatabase:
    """Simulated weather database with predefined city data."""
    
    # Base weather data for cities (temperature in Celsius)
    CITY_DATA = {
        "new york": {"country": "USA", "base_temp": 15, "conditions": ["Sunny", "Cloudy", "Rainy", "Partly Cloudy"]},
        "london": {"country": "UK", "base_temp": 12, "conditions": ["Cloudy", "Rainy", "Foggy", "Overcast"]},
        "tokyo": {"country": "Japan", "base_temp": 18, "conditions": ["Sunny", "Humid", "Rainy", "Clear"]},
        "paris": {"country": "France", "base_temp": 14, "conditions": ["Sunny", "Cloudy", "Mild", "Pleasant"]},
        "sydney": {"country": "Australia", "base_temp": 22, "conditions": ["Sunny", "Hot", "Clear", "Warm"]},
        "mumbai": {"country": "India", "base_temp": 30, "conditions": ["Hot", "Humid", "Sunny", "Monsoon"]},
        "dubai": {"country": "UAE", "base_temp": 35, "conditions": ["Hot", "Sunny", "Clear", "Dry"]},
        "moscow": {"country": "Russia", "base_temp": 5, "conditions": ["Cold", "Snowy", "Cloudy", "Freezing"]},
        "berlin": {"country": "Germany", "base_temp": 10, "conditions": ["Cloudy", "Cool", "Rainy", "Mild"]},
        "beijing": {"country": "China", "base_temp": 16, "conditions": ["Hazy", "Sunny", "Cloudy", "Windy"]},
        "singapore": {"country": "Singapore", "base_temp": 28, "conditions": ["Humid", "Tropical", "Rainy", "Hot"]},
        "toronto": {"country": "Canada", "base_temp": 8, "conditions": ["Cold", "Snowy", "Cloudy", "Cool"]},
        "los angeles": {"country": "USA", "base_temp": 24, "conditions": ["Sunny", "Clear", "Warm", "Pleasant"]},
        "san francisco": {"country": "USA", "base_temp": 16, "conditions": ["Foggy", "Cool", "Mild", "Cloudy"]},
        "cairo": {"country": "Egypt", "base_temp": 32, "conditions": ["Hot", "Sunny", "Dry", "Clear"]},
    }
    
    @classmethod
    def get_weather(cls, city: str) -> Optional[WeatherData]:
        """Fetch simulated weather data for a city."""
        city_key = city.lower().strip()
        
        if city_key not in cls.CITY_DATA:
            return None
        
        data = cls.CITY_DATA[city_key]
        
        # Add some randomness to simulate real-time data
        temp_variation = random.uniform(-3, 3)
        temperature_c = round(data["base_temp"] + temp_variation, 1)
        temperature_f = round(temperature_c * 9/5 + 32, 1)
        
        return WeatherData(
            city=city.title(),
            country=data["country"],
            temperature_celsius=temperature_c,
            temperature_fahrenheit=temperature_f,
            condition=random.choice(data["conditions"]),
            humidity=random.randint(30, 90),
            wind_speed_kmh=round(random.uniform(5, 30), 1),
            timestamp=datetime.now()
        )
    
    @classmethod
    def get_available_cities(cls) -> List[str]:
        """Return list of cities with available weather data."""
        return [city.title() for city in cls.CITY_DATA.keys()]

## 4. Weather Query Agent

The main agent class that processes weather queries and returns results.

In [ ]:
class WeatherQueryAgent:
    """
    An AI agent that processes weather queries and fetches temperature data.
    
    The agent can:
    - Parse natural language queries about weather
    - Fetch temperature for specific cities
    - Compare temperatures between cities
    - Provide weather summaries
    """
    
    def __init__(self):
        self.query_history: List[Dict] = []
        self.cache: Dict[str, WeatherData] = {}
    
    def query(self, user_input: str) -> str:
        """
        Process a natural language weather query.
        
        Args:
            user_input: Natural language query about weather
            
        Returns:
            Response string with weather information
        """
        # Log the query
        self.query_history.append({
            "query": user_input,
            "timestamp": datetime.now()
        })
        
        # Parse the query to extract city names
        cities = self._extract_cities(user_input)
        
        if not cities:
            return self._handle_no_city_found(user_input)
        
        # Fetch weather for each city
        results = []
        for city in cities:
            weather = self.get_temperature(city)
            if weather:
                results.append(weather)
            else:
                results.append(f"Sorry, I don't have weather data for '{city}'.")
        
        return "\n\n".join(str(r) for r in results)
    
    def get_temperature(self, city: str) -> Optional[WeatherData]:
        """
        Fetch temperature for a specific city.
        
        Args:
            city: Name of the city
            
        Returns:
            WeatherData object or None if city not found
        """
        weather = WeatherDatabase.get_weather(city)
        if weather:
            self.cache[city.lower()] = weather
        return weather
    
    def _extract_cities(self, text: str) -> List[str]:
        """Extract city names from natural language text."""
        available_cities = [c.lower() for c in WeatherDatabase.get_available_cities()]
        text_lower = text.lower()
        
        found_cities = []
        for city in available_cities:
            if city in text_lower:
                found_cities.append(city)
        
        return found_cities
    
    def _handle_no_city_found(self, query: str) -> str:
        """Handle queries where no city was identified."""
        if "help" in query.lower() or "cities" in query.lower():
            cities = WeatherDatabase.get_available_cities()
            return f"Available cities: {', '.join(cities)}"
        
        return (
            "I couldn't identify a city in your query. "
            "Please specify a city name. For example: 'What's the weather in Tokyo?'\n"
            f"Type 'help' or 'cities' to see available cities."
        )
    
    def compare_cities(self, city1: str, city2: str) -> str:
        """
        Compare temperatures between two cities.
        
        Args:
            city1: First city name
            city2: Second city name
            
        Returns:
            Comparison result string
        """
        weather1 = self.get_temperature(city1)
        weather2 = self.get_temperature(city2)
        
        if not weather1:
            return f"Sorry, I don't have weather data for '{city1}'."
        if not weather2:
            return f"Sorry, I don't have weather data for '{city2}'."
        
        diff = weather1.temperature_celsius - weather2.temperature_celsius
        
        if abs(diff) < 1:
            comparison = f"{weather1.city} and {weather2.city} have similar temperatures."
        elif diff > 0:
            comparison = f"{weather1.city} is {abs(diff):.1f}°C warmer than {weather2.city}."
        else:
            comparison = f"{weather1.city} is {abs(diff):.1f}°C colder than {weather2.city}."
        
        return (
            f"Temperature Comparison:\n"
            f"  {weather1.city}: {weather1.temperature_celsius}°C ({weather1.condition})\n"
            f"  {weather2.city}: {weather2.temperature_celsius}°C ({weather2.condition})\n"
            f"  {comparison}"
        )
    
    def get_query_history(self) -> List[Dict]:
        """Return the history of queries made to this agent."""
        return self.query_history

## 5. Demonstration

Let's see the Weather Query Agent in action!

In [ ]:
# Create the weather agent
agent = WeatherQueryAgent()

print("Weather Query Agent initialized!")
print("="*50)

### 5.1 Basic Temperature Queries

In [ ]:
# Query 1: Simple city query
print("Query: What's the weather in Tokyo?")
print("-" * 40)
result = agent.query("What's the weather in Tokyo?")
print(result)

In [ ]:
# Query 2: Multiple cities in one query
print("\nQuery: Tell me about the weather in London and Paris")
print("-" * 40)
result = agent.query("Tell me about the weather in London and Paris")
print(result)

### 5.2 Direct Temperature Lookup

In [ ]:
# Direct temperature lookup for a city
print("Direct lookup for New York:")
print("-" * 40)
weather = agent.get_temperature("New York")
if weather:
    print(f"City: {weather.city}")
    print(f"Temperature: {weather.temperature_celsius}°C / {weather.temperature_fahrenheit}°F")
    print(f"Condition: {weather.condition}")

### 5.3 Compare Temperatures Between Cities

In [ ]:
# Compare temperatures between two cities
print("Temperature Comparison: Dubai vs Moscow")
print("-" * 40)
comparison = agent.compare_cities("Dubai", "Moscow")
print(comparison)

In [ ]:
# Compare similar temperature cities
print("\nTemperature Comparison: London vs Berlin")
print("-" * 40)
comparison = agent.compare_cities("London", "Berlin")
print(comparison)

### 5.4 Help and Available Cities

In [ ]:
# Get list of available cities
print("Query: What cities are available?")
print("-" * 40)
result = agent.query("What cities are available?")
print(result)

### 5.5 Error Handling

In [ ]:
# Query for an unknown city
print("Query: What's the weather in Atlantis?")
print("-" * 40)
result = agent.query("What's the weather in Atlantis?")
print(result)

In [ ]:
# Query without a city name
print("\nQuery: Is it hot outside?")
print("-" * 40)
result = agent.query("Is it hot outside?")
print(result)

### 5.6 Query History

In [ ]:
# View query history
print("Agent Query History:")
print("=" * 50)
for i, entry in enumerate(agent.get_query_history(), 1):
    print(f"{i}. [{entry['timestamp'].strftime('%H:%M:%S')}] {entry['query']}")

## 6. Interactive Mode

Run the cell below to start an interactive session with the weather agent.

In [ ]:
def interactive_weather_agent():
    """Run an interactive weather query session."""
    agent = WeatherQueryAgent()
    
    print("="*60)
    print("      WEATHER QUERY AGENT - Interactive Mode")
    print("="*60)
    print("\nWelcome! Ask me about the weather in any city.")
    print("Type 'cities' to see available cities.")
    print("Type 'compare <city1> <city2>' to compare temperatures.")
    print("Type 'quit' or 'exit' to end the session.")
    print("-"*60)
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
            
            if not user_input:
                continue
            
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\nThank you for using the Weather Query Agent. Goodbye!")
                break
            
            # Handle compare command
            if user_input.lower().startswith('compare '):
                parts = user_input[8:].split(' and ')
                if len(parts) == 2:
                    print(f"\nAgent: {agent.compare_cities(parts[0].strip(), parts[1].strip())}")
                else:
                    parts = user_input[8:].split()
                    if len(parts) >= 2:
                        print(f"\nAgent: {agent.compare_cities(parts[0], parts[-1])}")
                    else:
                        print("\nAgent: Please specify two cities to compare.")
                continue
            
            # Regular query
            response = agent.query(user_input)
            print(f"\nAgent: {response}")
            
        except KeyboardInterrupt:
            print("\n\nSession interrupted. Goodbye!")
            break
        except EOFError:
            print("\nEnd of input. Goodbye!")
            break

# Uncomment the line below to start interactive mode
# interactive_weather_agent()

## 7. Summary

This notebook demonstrated a Weather Query Agent with the following capabilities:

1. **Natural Language Processing**: Parse queries to extract city names
2. **Temperature Fetching**: Retrieve simulated weather data for 15+ cities worldwide
3. **City Comparison**: Compare temperatures between different cities
4. **Query History**: Track all queries made to the agent
5. **Error Handling**: Gracefully handle unknown cities and unclear queries
6. **Interactive Mode**: Allow real-time conversation with the agent

### Supported Cities

The agent supports weather queries for:
- Americas: New York, Los Angeles, San Francisco, Toronto
- Europe: London, Paris, Berlin, Moscow
- Asia: Tokyo, Beijing, Mumbai, Singapore, Dubai
- Other: Sydney, Cairo

### Future Enhancements

Potential improvements for the agent:
- Integration with real weather APIs (OpenWeatherMap, WeatherAPI)
- Support for weather forecasts
- More sophisticated NLP for query understanding
- Weather alerts and notifications
- Historical weather data analysis